# 01 - Explore the raw APRA files

**Goal:** understand the raw files before cleaning anything. Real government Excel files have title rows,
merged headers, unit rows and footnotes. If we load them blindly, pandas reads junk.

Files (in `data/raw/`, never edited by hand):
- `fund_level_back_series_2004_2025.xlsx` - APRA Annual fund-level superannuation statistics, back series (June 2014 to June 2025 in the fund tables)
- `bulletin_entities_2015_2025.xlsx` - APRA Annual superannuation bulletin (industry-level totals), used later for reconciliation

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

FUND_FILE = "../data/raw/fund_level_back_series_2004_2025.xlsx"
BULLETIN_FILE = "../data/raw/bulletin_entities_2015_2025.xlsx"

## 1. What sheets exist?

In [2]:
xl = pd.ExcelFile(FUND_FILE)
xl.sheet_names

['Cover',
 'Notes',
 'Contents',
 'Important Notice',
 'Table 1',
 'Table 2a',
 'Table 2b',
 'Table 3a',
 'Table 3b',
 'Table 4',
 'Table 5',
 'Table 6',
 'Table 7',
 'Table 8',
 'Table 9',
 'Table 10',
 'Table 11',
 'Table 12',
 'Table 13',
 'Revisions',
 'Explanatory Notes']

`Table 2a` = fund profile for funds with a **30 June** year end. `Table 2b` = the same for funds with a **different** year end.
We need **both**, otherwise some funds go missing (this matters in the reconciliation later).

## 2. Find the real header row
Load the top rows with `header=None` so pandas does not guess.

In [3]:
pd.read_excel(FUND_FILE, sheet_name="Table 2a", header=None, nrows=9).iloc[:, :16]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table 2: Fund-level profile and structure,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A,B,C
4,Period,Fund name,ABN,RSE Regulatory classification,Fund type,RSE Membership base,Fund's RSE licensee,RSE licensee ownership type,RSE licensee profit status,RSE licensee board structure,RSE Balance date,Duration,RSE Wind up date,Total assets,Proportion of benefits which are defined benefits,Number of investment options*
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Financial flow duration only,NaN,SRF 320.0\nitem 11,Refer to Table 11 for details of ratio calcula...,SRF 601.0\nitem 6.1
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(months),NaN,($'000),(%),NaN
7,2025-06-30 00:00:00,AMG Super,30099320583,Public offer,Retail,General base,Equity Trustees Superannuation Limited,Financial services corporation ownership,For profit status,Non equal representation,2025-06-30 00:00:00,12,NaN,2794884,0,NaN
8,2025-06-30 00:00:00,AMP Super Fund,78421957449,Public offer,Retail,General base,N. M. Superannuation Proprietary Limited,Financial services corporation ownership,For profit status,Non equal representation,2025-06-30 00:00:00,12,NaN,59802440,0.01,NaN


What the top of the sheet shows:
- Rows 0-3: title and blank rows.
- **Row 4: the real column headers** -> `header=4`.
- Row 5: the APRA reporting form item (e.g. `SRF 320.0 item 11`). Not data.
- Row 6: **units**, e.g. `($'000)` and `(%)`. Not data.
- Data starts at row 7.

So after loading with `header=4` we must drop the first 2 rows (form item + units).

In [4]:
raw = pd.read_excel(FUND_FILE, sheet_name="Table 2a", header=4)
units = raw.iloc[1]                      # the units row, kept for the data dictionary
print(raw.shape)
pd.DataFrame({"column": raw.columns, "unit": units.values})

(1762, 28)


,column,unit
0,Period,NaN
1,Fund name,NaN
2,ABN,NaN
3,RSE Regulatory classification,NaN
4,Fund type,NaN
5,RSE Membership base,NaN
6,Fund's RSE licensee,NaN
7,RSE licensee ownership type,NaN
8,RSE licensee profit status,NaN
9,RSE licensee board structure,NaN


In [5]:
df_raw = raw.iloc[2:].copy()
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1760 entries, 2 to 1761
Data columns (total 28 columns):
 #   Column                                                      Non-Null Count  Dtype         
---  ------                                                      --------------  -----         
 0   Period                                                      1759 non-null   object        
 1   Fund name                                                   1758 non-null   str           
 2   ABN                                                         1758 non-null   float64       
 3   RSE Regulatory classification                               1758 non-null   str           
 4   Fund type                                                   1758 non-null   str           
 5   RSE Membership base                                         1758 non-null   str           
 6   Fund's RSE licensee                                         1758 non-null   str           
 7   RSE licensee ownership type        

## 3. Why are numeric columns loaded as text (`object`)?

In [6]:
def non_numeric_values(s):
    """Values in a column that cannot be read as numbers."""
    bad = s[pd.to_numeric(s, errors="coerce").isna() & s.notna()]
    return bad.astype(str).value_counts().to_dict()

for c in ["Total assets", "Total number of member accounts", "One-year\nrate of return", "Operating expense ratio"]:
    print(repr(c), "->", non_numeric_values(df_raw[c]))

'Total assets' -> {'*': 169}
'Total number of member accounts' -> {'*': 139}
'One-year\nrate of return' -> {'*': 156}
'Operating expense ratio' -> {'*': 156}


APRA's explanatory notes: *"To protect the privacy of individual members, APRA has masked certain items...
this is highlighted with an ' * '. Items which are blank indicate that nothing was reported or the data cannot be calculated."*

So `*` = **masked on purpose** and blank = **not reported**. These are different, and we will track them separately.

## 4. What does the Period column contain?

In [7]:
df_raw["Period"].astype(str).str[:60].value_counts().sort_index()

Period
*Starting from June 2023 period, the Annual fund-level stati      1
2014-06-30 00:00:00                                             216
2015-06-30 00:00:00                                             207
2016-06-30 00:00:00                                             194
2017-06-30 00:00:00                                             179
2018-06-30 00:00:00                                             170
2019-06-30 00:00:00                                             155
2020-06-30 00:00:00                                             143
2021-06-30 00:00:00                                             128
2022-06-30 00:00:00                                             109
2023-06-30 00:00:00                                             100
2024-06-30 00:00:00                                              85
2025-06-30 00:00:00                                              72
Name: count, dtype: int64

One value is not a date: a **footnote row** at the bottom of the sheet. It must be removed during cleaning.

In [8]:
df_raw["Fund type"].value_counts(dropna=False)

Fund type
Retail           968
Industry         375
Public Sector    190
Corporate        168
Retail - ERF      57
NaN                2
Name: count, dtype: int64

`Retail - ERF` = Eligible Rollover Fund (a retail fund type that held lost/inactive accounts). The bulletin has no
separate ERF line, so for reconciliation we group it under Retail.

In [9]:
rr = pd.to_numeric(df_raw["One-year\nrate of return"], errors="coerce")
rr.describe()

count    1567.000000
mean        0.060693
std         0.090816
min        -0.856000
25%         0.028000
50%         0.073000
75%         0.097000
max         0.889000
Name: One-year\nrate of return, dtype: float64

Returns are stored as **decimals** (median 0.073 = 7.3%), not as 7.3. That decides the range check later:
the bounds must be -0.5 to 0.5, not -50 to 50.

## Observations
1. Headers are on row 4; rows 5-6 are form references and units. Money is in **$ thousand**.
2. Numeric columns load as text because of `*` privacy masks.
3. A footnote row sits at the bottom of the sheet with text in the Period column.
4. Rates (returns, expense ratios) are decimals.
5. Fund count falls every year (shown in notebook 03), which is industry consolidation, not a data error.